# Mixup and CutMix Augmentation

**Difficulty**: 🟢 Easy

### Problem Statement

Mixup (Zhang et al., 2018) and CutMix (Yun et al., 2019) are part of essentially every modern image-classification training recipe: DeiT, Swin and ConvNeXt use Mixup with $\alpha = 0.8$ and CutMix with $\alpha = 1.0$, picking one of the two per batch, and timm's "ResNet strikes back" recipes use them too. Each image in the batch is paired with another image of the same batch, the network trains on a blend of the two, and the label is blended in the same proportion. This regularises the classifier and improves calibration and robustness.

Both are **batch-level** augmentations: they run on the collated batch (often on the GPU) rather than in `Dataset.__getitem__`, because every output needs two samples. Both need **soft labels**, so the loss is cross-entropy with probability targets.

**Mixup.** Draw one $\lambda \sim \mathrm{Beta}(\alpha, \alpha)$ and one random permutation $\pi$ of the batch:
$$\tilde x_i = \lambda\, x_i + (1-\lambda)\, x_{\pi(i)}, \qquad \tilde y_i = \lambda\, y_i + (1-\lambda)\, y_{\pi(i)}.$$

**CutMix.** Draw $\lambda \sim \mathrm{Beta}(\alpha, \alpha)$ and a box of height $H\sqrt{1-\lambda}$ and width $W\sqrt{1-\lambda}$ (so its area is a fraction $1-\lambda$ of the image), with its centre drawn uniformly over the image. Clip the box to the image and paste that region of $x_{\pi(i)}$ into $x_i$. Clipping shrinks boxes near the border, so **recompute**
$$\lambda = 1 - \frac{\text{area of the pasted box}}{H\,W}$$
and mix the labels with this $\lambda$, so the label matches the pixels the network actually sees.

---

### Requirements

1. **`mixup(x, y, alpha, generator=None) -> (x_mixed, y_mixed, lam)`**
   - `x`: `(N, C, H, W)` float images; `y`: `(N, K)` one-hot or soft labels whose rows sum to 1.
   - Draw **one** `lam ~ Beta(alpha, alpha)` and **one** permutation `perm = torch.randperm(N, generator=generator)` for the whole batch; return `lam * x + (1 - lam) * x[perm]`, `lam * y + (1 - lam) * y[perm]` and `lam` as a Python float.
2. **`cutmix(x, y, alpha, generator=None) -> (x_mixed, y_mixed, lam)`**
   - Same inputs. Draw `lam` and `perm` as above, then **one box for the whole batch**: `cut_h = int(H * sqrt(1 - lam))`, `cut_w = int(W * sqrt(1 - lam))`, centre `cy` uniform in `{0, …, H-1}` and `cx` uniform in `{0, …, W-1}`, edges `cy - cut_h // 2` to `cy - cut_h // 2 + cut_h` (likewise for x), clipped to `[0, H]` and `[0, W]`.
   - Paste `x[perm]`'s box into a copy of `x`, recompute `lam = 1 - box_area / (H * W)` from the clipped box, and mix the labels with that `lam`. Return the recomputed `lam`.
   - Exact rounding of the box size is up to you; what is not optional is that every pixel of `x_mixed[i]` comes from `x[i]` or `x[perm[i]]`, and that the fraction taken from the partner equals `1 - lam`.
3. **Reproducibility.** Draw *all* randomness — `lam`, the permutation and the box centre — from `generator`; when `generator is None`, use the global RNG (`torch.manual_seed`). An explicit generator makes a data pipeline replayable (re-running step 41,000 to debug a loss spike), lets each dataloader worker or GPU rank own a seeded stream, and keeps augmentation from consuming the global RNG that dropout and initialisation also depend on.
4. **Sampling Beta with a generator.** `torch.distributions.Beta(a, a).sample()` takes no generator. Use $\mathrm{Beta}(a, b) = \frac{G_a}{G_a + G_b}$ with independent $G_a \sim \mathrm{Gamma}(a)$, $G_b \sim \mathrm{Gamma}(b)$; `torch._standard_gamma(torch.tensor([a, a], dtype=torch.float64), generator=g)` draws both Gammas from your generator (it is the op `torch.distributions.Gamma` calls internally). Use float64: for small $\alpha$ a float32 Gamma draw can underflow to 0.
5. **The role of $\alpha$.** $\alpha \to 0$: $\mathrm{Beta}(\alpha, \alpha)$ piles up at 0 and 1, so most batches are barely mixed (plain training). $\alpha = 1$: $\lambda$ is uniform on $[0, 1]$. $\alpha \to \infty$: $\lambda \to 0.5$. **`alpha <= 0` disables mixing**: return copies of `x` and `y` and `lam = 1.0`.
6. **Do not modify `x` or `y` in place.** The caller may still need the clean batch (logging, a teacher model, another augmentation).

---

### Constraints

- ✅ Pure PyTorch, vectorized over the batch (one indexing op with `perm`, one slice assignment for the box).
- ❌ No NumPy or Python `random` (they would ignore `generator`), and no `torch.distributions.Beta(...).sample()` (it has no generator argument).
- ❌ No `torchvision.transforms.v2.MixUp` / `CutMix` or timm's `Mixup`.
- ❌ No Python loop over the samples of the batch.

---

<details>
  <summary>💡 Hint</summary>

  - Mixing with a permuted copy of the batch, `x[perm]`, pairs every sample with a partner in one indexing op. The labels must use the same `perm` and the same `lam`.
  - `torch.randperm`, `torch.randint` and `torch.rand` all accept `generator=`.
  - CutMix's box shares the image's aspect ratio: scale both sides by the same factor. What factor gives an area fraction of $1 - \lambda$?
  - After clipping, the box you pasted may be smaller than the box you planned. Which $\lambda$ describes the image the network sees?
  - `x.clone()` before writing the box, or build the output with a mask: `x * (1 - m) + x[perm] * m`.

</details>

---

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
N, C, H, W, K = 8, 3, 12, 20, 10
gen = torch.Generator().manual_seed(0)
x = torch.randn(N, C, H, W, generator=gen)       # a batch of "images" (non-square on purpose)
labels = torch.randperm(K, generator=gen)[:N]    # distinct classes, so pairings can be read off
y = F.one_hot(labels, K).float()                 # (N, K) one-hot targets
print("x:", tuple(x.shape), " y:", tuple(y.shape), " classes:", labels.tolist())


def partner_of(y_mixed, labels):
    """Read the pairing off mixed one-hot labels: perm[i] is the sample whose class appears in
    row i besides i's own (i itself if none does). Valid only when all classes are distinct."""
    where = {int(c): i for i, c in enumerate(labels)}
    perm = []
    for i, c in enumerate(labels.tolist()):
        row = y_mixed[i].clone()
        row[c] = 0
        perm.append(where[int(row.argmax())] if row.max() > 0 else i)
    return torch.tensor(perm)

In [ ]:
def mixup(x: torch.Tensor, y: torch.Tensor, alpha: float, generator=None):
    """
    Mixup one batch with a random permutation of itself.

    Args:
        x: (N, C, H, W) float images.
        y: (N, K) one-hot or soft labels, rows sum to 1.
        alpha: Beta(alpha, alpha) concentration; alpha <= 0 disables mixing.
        generator: torch.Generator for ALL randomness; None -> global RNG.

    Returns:
        x_mixed: (N, C, H, W) = lam * x + (1 - lam) * x[perm]
        y_mixed: (N, K)       = lam * y + (1 - lam) * y[perm]
        lam: float
    """
    # TODO: lam ~ Beta(alpha, alpha) from the generator (two Gamma draws), perm = randperm(N)
    raise NotImplementedError


def cutmix(x: torch.Tensor, y: torch.Tensor, alpha: float, generator=None):
    """
    CutMix one batch with a random permutation of itself (one box for the whole batch).

    Args:
        x: (N, C, H, W) float images.
        y: (N, K) one-hot or soft labels, rows sum to 1.
        alpha: Beta(alpha, alpha) concentration; alpha <= 0 disables mixing.
        generator: torch.Generator for ALL randomness; None -> global RNG.

    Returns:
        x_mixed: (N, C, H, W) — x with one box replaced by the same box of x[perm]
        y_mixed: (N, K)       = lam * y + (1 - lam) * y[perm]
        lam: float            = 1 - (area of the clipped box) / (H * W)
    """
    # TODO: lam and perm as in mixup; box sides H*sqrt(1-lam), W*sqrt(1-lam); uniform centre;
    #       clip; paste into a clone; recompute lam from the clipped box; mix labels
    raise NotImplementedError

In [ ]:
# Test 1: shapes, lam in [0, 1], label rows still sum to 1
for fn in (mixup, cutmix):
    xm, ym, lam = fn(x, y, 1.0, generator=torch.Generator().manual_seed(1))
    assert xm.shape == x.shape and ym.shape == y.shape, f"{fn.__name__}: shapes changed"
    assert 0.0 <= float(lam) <= 1.0, f"{fn.__name__}: lam = {float(lam)} outside [0, 1]"
    assert torch.allclose(ym.sum(1), torch.ones(N), atol=1e-6), f"{fn.__name__}: label rows must sum to 1"
print("Test 1 passed: shapes and label rows")

In [ ]:
# Test 2: mixup is exactly lam * x + (1 - lam) * x[perm] for a permutation perm
for seed in range(5):
    xm, ym, lam = mixup(x, y, 0.4, generator=torch.Generator().manual_seed(seed))
    perm = partner_of(ym, labels)
    assert sorted(perm.tolist()) == list(range(N)), f"partners {perm.tolist()} are not a permutation"
    assert torch.allclose(xm, lam * x + (1 - lam) * x[perm], atol=1e-6), "images and labels disagree on lam/perm"
    assert torch.allclose(ym, lam * y + (1 - lam) * y[perm], atol=1e-6)
print("Test 2 passed: mixup blends each sample with its partner")

In [ ]:
# Test 3: every CutMix pixel comes from the image or its partner, and the partner's share is 1 - lam
for seed in range(20):
    xm, ym, lam = cutmix(x, y, 1.0, generator=torch.Generator().manual_seed(seed))
    perm = partner_of(ym, labels)
    assert ((xm == x) | (xm == x[perm])).all(), "CutMix copies pixels; it never blends them"
    from_partner = (xm == x[perm]) & (xm != x)          # all False for samples paired with themselves
    for i in (perm != torch.arange(N)).nonzero().flatten().tolist():
        frac = from_partner[i, 0].float().mean().item()
        assert abs(frac - (1 - lam)) < 1e-6, \
            f"seed {seed}: {frac:.4f} of sample {i}'s pixels come from its partner but 1 - lam = {1 - lam:.4f}"
    assert torch.allclose(ym, lam * y + (1 - lam) * y[perm], atol=1e-6)

print("Test 3 passed: CutMix pastes the partner's box and lam is its true share")
partnered = (perm != torch.arange(N)).nonzero().flatten().tolist()
if partnered:
    i = partnered[0]
    print(f"Last call: lam = {lam:.3f}; sample {i} took the '#' box from sample {int(perm[i])}:")
    for r in range(H):
        print("   " + "".join("#" if v else "." for v in from_partner[i, 0, r].tolist()))

In [ ]:
# Test 4: same generator seed -> same result (even if the global RNG moved), alpha=0 is a no-op,
# and the inputs are not modified in place
x_before, y_before = x.clone(), y.clone()
for fn in (mixup, cutmix):
    a = fn(x, y, 1.0, generator=torch.Generator().manual_seed(7))
    torch.rand(5)                                         # disturb the global RNG
    b = fn(x, y, 1.0, generator=torch.Generator().manual_seed(7))
    assert torch.equal(a[0], b[0]) and torch.equal(a[1], b[1]) and float(a[2]) == float(b[2]), \
        f"{fn.__name__}: not reproducible from the generator"
    xm, ym, lam = fn(x, y, 0.0)
    assert float(lam) == 1.0 and torch.equal(xm, x) and torch.equal(ym, y), f"{fn.__name__}: alpha=0 should not mix"
assert torch.equal(x, x_before) and torch.equal(y, y_before), "x or y was modified in place"
print("Test 4 passed: reproducible, alpha=0 is a no-op, inputs untouched")

In [ ]:
# Test 5 (look): how alpha shapes the distribution of lam
for alpha in (0.2, 1.0, 10.0):
    lams = torch.tensor([float(mixup(x[:2], y[:2], alpha, generator=torch.Generator().manual_seed(s))[2])
                         for s in range(300)])
    edge = ((lams < 0.1) | (lams > 0.9)).float().mean().item()
    print(f"alpha={alpha:>4}: mean lam {lams.mean():.2f}, std {lams.std():.2f}, within 0.1 of 0 or 1: {edge:.0%}")

In [ ]:
# Test 6: cross-entropy with mixed soft labels == the same mix of the two hard-label losses
# (CE is linear in the target), which is why some implementations mix losses instead of labels.
logits = torch.randn(N, K)
xm, ym, lam = mixup(x, y, 1.0, generator=torch.Generator().manual_seed(0))
perm = partner_of(ym, labels)
soft = F.cross_entropy(logits, ym)
mixed = lam * F.cross_entropy(logits, labels) + (1 - lam) * F.cross_entropy(logits, labels[perm])
assert torch.allclose(soft, mixed, atol=1e-5)
print(f"Test 6 passed: CE(soft) = {soft.item():.4f} = lam*CE(y) + (1-lam)*CE(y[perm]) = {mixed.item():.4f}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("mixup-cutmix", mixup=mixup, cutmix=cutmix)